# Q7 — Diagnose, Redesign and Validate

Uses the Q5 fully connected classifier's translation sensitivity as a candidate bottleneck and tests a CNN redesign under matched conditions. The measured results determine whether the hypothesis is supported.

## 1. Purpose and selected bottleneck

This notebook uses the **translation sensitivity of the fully connected image classifier from Q5** as the candidate limitation. A fully connected classifier receives flattened pixels and has no built-in spatial weight sharing. The redesign is a small CNN with local shared filters and pooling.

Run Q5 first if you want the exact same sample split and training settings as your Q5 experiment. This notebook independently recreates the same Fashion-MNIST subset, seed, model pair and training budget so it can also run on its own. The comparison is original FC model versus redesigned CNN.

In [ ]:
# 1. Imports and setup
import os,time,random,json,zipfile,gc
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
import torch,torch.nn as nn,torch.nn.functional as F
from torch.utils.data import DataLoader,Subset
from torchvision import datasets,transforms
from sklearn.metrics import accuracy_score

SEED=42; EPOCHS=3; BATCH=128; LR=0.001
if not torch.cuda.is_available(): torch.set_num_threads(min(4,os.cpu_count() or 1))
def seed_all():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
seed_all()
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT=Path("Q7_outputs"); OUT.mkdir(exist_ok=True)
print("Device:",device)


## 2. Data and controlled conditions

Both models use the same Fashion-MNIST examples, seed, optimizer, learning rate, batch size and epoch budget. The translated test set is created by applying the same fixed pixel shift to every test image.

In [ ]:
tf=transforms.ToTensor()
train_full=datasets.FashionMNIST("./data",train=True,download=True,transform=tf)
test_full=datasets.FashionMNIST("./data",train=False,download=True,transform=tf)
g=torch.Generator().manual_seed(SEED); perm=torch.randperm(len(train_full),generator=g)
TRAIN_N,VAL_N,TEST_N=5000,1000,2000
train_ds=Subset(train_full,perm[:TRAIN_N].tolist())
val_ds=Subset(train_full,perm[TRAIN_N:TRAIN_N+VAL_N].tolist())
test_ds=Subset(test_full,list(range(TEST_N)))
def make_loaders():
    gen=torch.Generator().manual_seed(SEED)
    return (DataLoader(train_ds,batch_size=BATCH,shuffle=True,num_workers=0,generator=gen),
            DataLoader(val_ds,batch_size=BATCH,shuffle=False,num_workers=0),
            DataLoader(test_ds,batch_size=BATCH,shuffle=False,num_workers=0))


## 3. Original and redesigned models

**Original:** fully connected network operating on flattened 784-pixel input.

**Redesign:** convolutional network using local receptive fields, shared filters and max pooling. The redesign adds a spatial inductive bias; it is expected to be less sensitive to small position changes, but the actual change must be measured.

In [ ]:
class OriginalFC(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Flatten(),nn.Linear(784,128),nn.ReLU(),nn.Linear(128,10))
    def forward(self,x): return self.net(x)

class RedesignedCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1=nn.Conv2d(1,16,3,padding=1)
        self.c2=nn.Conv2d(16,32,3,padding=1)
        self.pool=nn.MaxPool2d(2)
        self.fc=nn.Linear(32*7*7,10)
    def forward(self,x):
        x=self.pool(F.relu(self.c1(x)))
        x=self.pool(F.relu(self.c2(x)))
        return self.fc(torch.flatten(x,1))

models={"Original_FC":OriginalFC,"Redesigned_CNN":RedesignedCNN}
for name,factory in models.items():
    m=factory()
    print(name,"parameters:",sum(p.numel() for p in m.parameters()))


## 4. Pre-implementation design objective and prediction

**Objective:** improve the translated-test accuracy of the original classifier while maintaining comparable clean-test accuracy under the same training budget.

**Prediction:** the CNN should show a smaller accuracy drop under a small image translation because the same convolutional filters are applied at every spatial location and pooling reduces sensitivity to small local shifts. This is a hypothesis, not a guaranteed outcome.

In [ ]:
# Record the pre-experiment objective and prediction as part of the evidence.
design_statement={
 "objective":"Reduce the accuracy drop under a fixed 3-pixel right and 2-pixel down translation while retaining clean-test accuracy.",
 "prediction":"The CNN is expected to have a smaller translated-test accuracy drop than the flattened fully connected network.",
 "comparison":"Same data split, seed, optimizer, learning rate, batch size and epoch budget."
}
with open(OUT/"Q7_pre_experiment_prediction.json","w") as f: json.dump(design_statement,f,indent=2)
print(design_statement)


## 5. Training and evaluation functions

The test set is not used to select checkpoints or tune training. We evaluate clean and translated test examples only after each model finishes training.

In [ ]:
criterion=nn.CrossEntropyLoss()
def translate_batch(x,dx=3,dy=2):
    theta=torch.tensor([[1,0,-2*dx/28],[0,1,-2*dy/28]],dtype=x.dtype,device=x.device)
    theta=theta.unsqueeze(0).repeat(x.size(0),1,1)
    grid=F.affine_grid(theta,x.size(),align_corners=False)
    return F.grid_sample(x,grid,mode="nearest",padding_mode="zeros",align_corners=False)

def eval_loader(model,loader,translated=False):
    model.eval(); loss_sum=0; pred=[]; actual=[]
    with torch.no_grad():
        for x,y in loader:
            x,y=x.to(device),y.to(device)
            if translated: x=translate_batch(x)
            z=model(x); loss_sum+=criterion(z,y).item()*len(y)
            pred.extend(z.argmax(1).cpu().tolist()); actual.extend(y.cpu().tolist())
    return loss_sum/len(loader.dataset),accuracy_score(actual,pred)

def train_model(model,tr,va):
    opt=torch.optim.Adam(model.parameters(),lr=LR); hist=[]
    if device.type=="cuda": torch.cuda.synchronize()
    start=time.perf_counter()
    for ep in range(1,EPOCHS+1):
        model.train(); total=0
        for x,y in tr:
            x,y=x.to(device),y.to(device); opt.zero_grad(set_to_none=True)
            loss=criterion(model(x),y); loss.backward(); opt.step(); total+=loss.item()*len(y)
        vl,va_acc=eval_loader(model,va)
        hist.append({"epoch":ep,"train_loss":total/len(tr.dataset),"val_loss":vl,"val_accuracy":va_acc})
        print(f"Epoch {ep}/{EPOCHS}: val loss={vl:.4f}, val acc={va_acc:.4f}")
    if device.type=="cuda": torch.cuda.synchronize()
    return hist,time.perf_counter()-start


## 6. Implement and validate the redesign

In [ ]:
results=[]; histories={}
for name,factory in models.items():
    print("\\nTraining",name); seed_all()
    tr,va,te=make_loaders()
    model=factory().to(device)
    hist,elapsed=train_model(model,tr,va)
    clean_loss,clean_acc=eval_loader(model,te,translated=False)
    shifted_loss,shifted_acc=eval_loader(model,te,translated=True)
    params=sum(p.numel() for p in model.parameters())
    results.append({"Model":name,"Parameters":params,"Training_Time_s":elapsed,
      "Clean_Test_Loss":clean_loss,"Clean_Test_Accuracy":clean_acc,
      "Shifted_Test_Loss":shifted_loss,"Shifted_Test_Accuracy":shifted_acc,
      "Accuracy_Drop":clean_acc-shifted_acc})
    histories[name]=hist
    torch.save(model.state_dict(),OUT/f"{name}.pth")
    pd.DataFrame(results).to_csv(OUT/"Q7_checkpoint.csv",index=False)
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

results_df=pd.DataFrame(results); display(results_df)
results_df.to_csv(OUT/"Q7_original_vs_redesign.csv",index=False)


## 7. Compare learning curves and trade-offs

In [ ]:
plt.figure(figsize=(8,4))
for name,h in histories.items():
    plt.plot([r["epoch"] for r in h],[r["train_loss"] for r in h],marker="o",label=f"{name} train")
    plt.plot([r["epoch"] for r in h],[r["val_loss"] for r in h],marker="s",linestyle="--",label=f"{name} val")
plt.xlabel("Epoch"); plt.ylabel("Cross-entropy loss"); plt.title("Original vs redesigned learning curves")
plt.grid(alpha=.3); plt.legend(); plt.tight_layout(); plt.savefig(OUT/"Q7_learning_curves.png",dpi=150); plt.show()

fig,ax=plt.subplots(1,2,figsize=(10,4))
ax[0].bar(results_df["Model"],results_df["Clean_Test_Accuracy"]); ax[0].set_title("Clean test accuracy"); ax[0].tick_params(axis="x",rotation=15)
ax[1].bar(results_df["Model"],results_df["Accuracy_Drop"]); ax[1].set_title("Accuracy drop after translation"); ax[1].tick_params(axis="x",rotation=15)
for a in ax: a.grid(axis="y",alpha=.25)
plt.tight_layout(); plt.savefig(OUT/"Q7_translation_comparison.png",dpi=150); plt.show()


## 8. Automated evidence summary

Use this table to assess whether the pre-registered objective was met. Do not assume the CNN wins: a redesign that fails to improve is still analysable if the result is reported and explained honestly.

In [ ]:
if len(results_df)==2:
    original=results_df[results_df["Model"]=="Original_FC"].iloc[0]
    redesigned=results_df[results_df["Model"]=="Redesigned_CNN"].iloc[0]
    summary={
      "clean_accuracy_change_CNN_minus_FC":float(redesigned["Clean_Test_Accuracy"]-original["Clean_Test_Accuracy"]),
      "translation_drop_change_CNN_minus_FC":float(redesigned["Accuracy_Drop"]-original["Accuracy_Drop"]),
      "parameter_change_CNN_minus_FC":int(redesigned["Parameters"]-original["Parameters"]),
      "training_time_change_seconds":float(redesigned["Training_Time_s"]-original["Training_Time_s"]),
      "objective_met_smaller_translation_drop":bool(redesigned["Accuracy_Drop"]<original["Accuracy_Drop"])
    }
    print(summary)
    with open(OUT/"Q7_comparison_summary.json","w") as f: json.dump(summary,f,indent=2)


## 9. Export evidence

In [ ]:
with open(OUT/"Q7_histories.json","w") as f: json.dump(histories,f,indent=2)
with open(OUT/"Q7_settings.json","w") as f:
    json.dump({"seed":SEED,"epochs":EPOCHS,"batch_size":BATCH,"learning_rate":LR,
               "train":TRAIN_N,"validation":VAL_N,"test":TEST_N,"device":str(device),
               "translation_pixels":{"right":3,"down":2}},f,indent=2)
with zipfile.ZipFile("Q7_complete_evidence.zip","w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file(): z.write(p,arcname=p.relative_to(OUT.parent))
print("Created Q7_complete_evidence.zip")
try:
    from google.colab import files
    files.download("Q7_complete_evidence.zip")
except ImportError: pass
